<a href="https://colab.research.google.com/github/hamza-24-ai/ca_research/blob/main/CA_TASK_03_LSTM_Inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Install DataSet Of WikiSales

In [1]:
!pip -q install datasets

import torch, torch.nn as nn, torch.optim as optim, time
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from collections import Counter

torch.manual_seed(0)
dev = torch.device("cuda")
WIN = 20
EPOCHS = 5

ds = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1")
def toks(split):
    out = []
    for line in ds[split]["text"]:
        out.extend(line.lower().split())
    return out
train_tok = toks("train"); val_tok = toks("validation")

# vocab from train only, <UNK>=0 (as in notebook)
vocab = {"<UNK>": 0}
for w, _ in Counter(train_tok).most_common():
    vocab[w] = len(vocab)
V = len(vocab)
print("vocab size:", V)

def windows(tokens):
    ids = [vocab.get(w, 0) for w in tokens]
    X = [ids[i:i+WIN] for i in range(len(ids)-WIN)]
    y = [ids[i+WIN] for i in range(len(ids)-WIN)]
    return torch.tensor(X), torch.tensor(y)

class DS(Dataset):
    def __init__(self, X, y): self.X, self.y = X, y
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

Xtr, ytr = windows(train_tok)
Xva, yva = windows(val_tok)
print("train windows:", len(Xtr), "| val windows:", len(Xva))

class LSTM(nn.Module):              # same architecture as the notebook
    def __init__(self, vocab_size):
        super().__init__()
        self.embedded = nn.Embedding(vocab_size, 100)
        self.lstm = nn.LSTM(100, 150, batch_first=True)
        self.linear = nn.Linear(150, vocab_size)
    def forward(self, x):
        emb = self.embedded(x)
        _, (final_hidden, _) = self.lstm(emb)
        return self.linear(final_hidden.squeeze(0))

model = LSTM(V).to(dev)
print("params:", sum(p.numel() for p in model.parameters()))
opt = optim.Adam(model.parameters(), lr=0.001)
crit = nn.CrossEntropyLoss()
dl = DataLoader(DS(Xtr, ytr), batch_size=32, shuffle=True)

t0 = time.time()
for ep in range(EPOCHS):
    model.train(); total = 0
    for xb, yb in dl:
        xb, yb = xb.to(dev), yb.to(dev)
        opt.zero_grad()
        loss = crit(model(xb), yb)
        loss.backward(); opt.step()
        total += loss.item()
    print(f"Epoch {ep+1} loss {total:.2f}")
print("train time (s):", round(time.time() - t0, 1))

# held-out accuracy (WikiText validation split)
model.eval(); correct = 0
with torch.no_grad():
    for i in range(0, len(Xva), 256):
        xb = Xva[i:i+256].to(dev); yb = yva[i:i+256].to(dev)
        correct += (model(xb).argmax(1) == yb).sum().item()
print("held-out top-1 accuracy:", correct / len(Xva))

torch.save({"state_dict": model.state_dict(), "vocab": vocab, "win": WIN}, "lstm_wikitext2.pt")
print("saved lstm_wikitext2.pt")

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…): reconstructing file:   0%|          |  0.00B /  733kB            

wikitext-2-raw-v1/test-00000-of-00001.pa(…): downloading bytes:           |  0.00B            

wikitext-2-raw-v1/train-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 6.36MB            

wikitext-2-raw-v1/train-00000-of-00001.p(…): downloading bytes:           |  0.00B            

wikitext-2-raw-v1/validation-00000-of-00(…): reconstructing file:   0%|          |  0.00B /  657kB            

wikitext-2-raw-v1/validation-00000-of-00(…): downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

vocab size: 66650
train windows: 2051890 | val windows: 213866
params: 16880350
Epoch 1 loss 411765.79
Epoch 2 loss 385076.64
Epoch 3 loss 370354.48
Epoch 4 loss 359387.56
Epoch 5 loss 350950.36
train time (s): 3036.1
held-out top-1 accuracy: 0.2070642364845277
saved lstm_wikitext2.pt


# CPU Inference Testing

In [2]:
import torch, torch.nn as nn, time, statistics, csv
from datasets import load_dataset
from collections import Counter

torch.manual_seed(0)
dev = torch.device("cpu")
WIN = 20
ck = torch.load("lstm_wikitext2.pt", map_location="cpu")
vocab = ck["vocab"]; V = len(vocab)
print("torch", torch.__version__, "| device:", dev, "| vocab:", V)

class LSTM(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embedded = nn.Embedding(vocab_size, 100)
        self.lstm = nn.LSTM(100, 150, batch_first=True)
        self.linear = nn.Linear(150, vocab_size)
    def forward(self, x):
        emb = self.embedded(x)
        _, (final_hidden, _) = self.lstm(emb)
        return self.linear(final_hidden.squeeze(0))

base = LSTM(V)
base.load_state_dict(ck["state_dict"])
base.eval()
print("params:", sum(p.numel() for p in base.parameters()))

# FLOPs per sample (analytic, MAC x 2)
# LSTM: 4*H*(I+H) per time step, x WIN steps; Linear: H*V
H, I = 150, 100
macs = 4 * H * (I + H) * WIN + H * V
flops_per_sample = 2 * macs
print("FLOPs/sample:", flops_per_sample)

def make_inputs(bs):
    return torch.randint(1, V, (bs, WIN))

def bench(m, name, dt, bs, reps):
    x = make_inputs(bs)
    with torch.inference_mode():
        for _ in range(3):
            m(x)
        ts = []
        for _ in range(reps):
            t0 = time.perf_counter(); m(x); ts.append(time.perf_counter() - t0)
    s = sorted(ts); med = statistics.median(ts)
    return {"device": "cpu", "precision": name, "batch": bs, "reps": reps,
            "median_s": med, "std_s": statistics.stdev(ts),
            "p95_s": s[int(0.95*(len(s)-1))], "p99_s": s[int(0.99*(len(s)-1))],
            "throughput_seq_s": bs / med,
            "gflops_per_s": flops_per_sample * bs / med / 1e9, "status": "ok"}

rows = []
for name, dt in [("fp32", torch.float32), ("bf16", torch.bfloat16)]:
    m = LSTM(V); m.load_state_dict(ck["state_dict"]); m.eval(); m = m.to(dt)
    for bs in [1, 2, 4, 8, 16, 32]:
        try:
            rows.append(bench(m, name, dt, bs, 100 if bs == 1 else 10))
        except Exception as e:
            rows.append({"device": "cpu", "precision": name, "batch": bs,
                         "status": f"failed: {type(e).__name__}: {str(e)[:100]}"})
        print(rows[-1])

# ---------- accuracy on fixed held-out subset ----------
val = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="validation")
vt = []
for line in val["text"]:
    vt.extend(line.lower().split())
ids = [vocab.get(w, 0) for w in vt]
X = torch.tensor([ids[i:i+WIN] for i in range(len(ids)-WIN)])
y = torch.tensor([ids[i+WIN] for i in range(len(ids)-WIN)])
Xs, ys = X[:10000], y[:10000]           # fixed first 10,000 windows
print("accuracy windows:", len(Xs))

for name, dt in [("fp32", torch.float32), ("bf16", torch.bfloat16)]:
    m = LSTM(V); m.load_state_dict(ck["state_dict"]); m.eval(); m = m.to(dt)
    correct = 0
    with torch.inference_mode():
        for i in range(0, len(Xs), 256):
            out = m(Xs[i:i+256]).float()          # input stays Long, only model is bf16
            correct += (out.argmax(1) == ys[i:i+256]).sum().item()
    acc = correct / len(Xs)
    rows.append({"device": "cpu", "precision": name, "batch": "accuracy",
                 "reps": len(Xs), "accuracy": acc, "status": "ok"})
    print(name, "accuracy:", acc)

with open("lstm_cpu.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=sorted({k for r in rows for k in r}))
    w.writeheader(); w.writerows(rows)
print("Saved lstm_cpu.csv")

torch 2.14.1+cpu | device: cpu | vocab: 66650
params: 16880350
FLOPs/sample: 25995000
{'device': 'cpu', 'precision': 'fp32', 'batch': 1, 'reps': 100, 'median_s': 0.006721406500020066, 'std_s': 0.0013957694800526411, 'p95_s': 0.008538591999979417, 'p99_s': 0.009567082999978993, 'throughput_seq_s': 148.7783844046651, 'gflops_per_s': 3.867494102599269, 'status': 'ok'}
{'device': 'cpu', 'precision': 'fp32', 'batch': 2, 'reps': 10, 'median_s': 0.009875993500003233, 'std_s': 0.001042323113740167, 'p95_s': 0.010289101000012124, 'p99_s': 0.010289101000012124, 'throughput_seq_s': 202.51127139759106, 'gflops_per_s': 5.26428049998038, 'status': 'ok'}
{'device': 'cpu', 'precision': 'fp32', 'batch': 4, 'reps': 10, 'median_s': 0.010631252500019173, 'std_s': 0.0005136454189972177, 'p95_s': 0.011475889999985611, 'p99_s': 0.011475889999985611, 'throughput_seq_s': 376.24917666030285, 'gflops_per_s': 9.780597347284573, 'status': 'ok'}
{'device': 'cpu', 'precision': 'fp32', 'batch': 8, 'reps': 10, 'median

# GPU iNFERENCE Testing

In [1]:
!pip -q install datasets

import torch, torch.nn as nn, time, statistics, csv
from datasets import load_dataset

torch.manual_seed(0)
dev = torch.device("cuda")
WIN = 20
ck = torch.load("lstm_wikitext2.pt", map_location="cpu")
vocab = ck["vocab"]; V = len(vocab)
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0), "| vocab:", V)

class LSTM(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embedded = nn.Embedding(vocab_size, 100)
        self.lstm = nn.LSTM(100, 150, batch_first=True)
        self.linear = nn.Linear(150, vocab_size)
    def forward(self, x):
        emb = self.embedded(x)
        _, (final_hidden, _) = self.lstm(emb)
        return self.linear(final_hidden.squeeze(0))

def load_model(dt):
    m = LSTM(V); m.load_state_dict(ck["state_dict"]); m.eval()
    return m.to(dev).to(dt)

print("params:", sum(p.numel() for p in LSTM(V).parameters()))

H, I = 150, 100
macs = 4 * H * (I + H) * WIN + H * V
flops_per_sample = 2 * macs
print("FLOPs/sample:", flops_per_sample)

def sync():
    torch.cuda.synchronize()

def make_inputs(bs):
    return torch.randint(1, V, (bs, WIN), device=dev)   # Long ids, stays Long

def bench(m, name, dt, bs, reps):
    x = make_inputs(bs)
    with torch.inference_mode():
        for _ in range(3):
            m(x)
        sync()
        ts = []
        for _ in range(reps):
            t0 = time.perf_counter(); m(x); sync(); ts.append(time.perf_counter() - t0)
    s = sorted(ts); med = statistics.median(ts)
    return {"device": "cuda", "precision": name, "batch": bs, "reps": reps,
            "median_s": med, "std_s": statistics.stdev(ts),
            "p95_s": s[int(0.95*(len(s)-1))], "p99_s": s[int(0.99*(len(s)-1))],
            "throughput_seq_s": bs / med,
            "gflops_per_s": flops_per_sample * bs / med / 1e9, "status": "ok"}

rows = []
for name, dt in [("fp32", torch.float32), ("fp16", torch.float16), ("bf16", torch.bfloat16)]:
    try:
        m = load_model(dt)
    except Exception as e:
        rows.append({"device": "cuda", "precision": name, "status": f"load failed: {type(e).__name__}: {str(e)[:100]}"})
        print(rows[-1]); continue
    for bs in [1, 2, 4, 8, 16, 32]:
        try:
            rows.append(bench(m, name, dt, bs, 100 if bs == 1 else 10))
        except Exception as e:
            rows.append({"device": "cuda", "precision": name, "batch": bs,
                         "status": f"failed: {type(e).__name__}: {str(e)[:100]}"})
        print(rows[-1])

# ---------- accuracy on the same 10,000 validation windows as CPU ----------
val = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="validation")
vt = []
for line in val["text"]:
    vt.extend(line.lower().split())
ids = [vocab.get(w, 0) for w in vt]
X = torch.tensor([ids[i:i+WIN] for i in range(len(ids)-WIN)])
y = torch.tensor([ids[i+WIN] for i in range(len(ids)-WIN)])
Xs, ys = X[:10000].to(dev), y[:10000].to(dev)
print("accuracy windows:", len(Xs))

for name, dt in [("fp32", torch.float32), ("fp16", torch.float16), ("bf16", torch.bfloat16)]:
    try:
        m = load_model(dt)
        correct = 0
        with torch.inference_mode():
            for i in range(0, len(Xs), 256):
                out = m(Xs[i:i+256]).float()
                correct += (out.argmax(1) == ys[i:i+256]).sum().item()
        acc = correct / len(Xs)
        rows.append({"device": "cuda", "precision": name, "batch": "accuracy",
                     "reps": len(Xs), "accuracy": acc, "status": "ok"})
        print(name, "accuracy:", acc)
    except Exception as e:
        rows.append({"device": "cuda", "precision": name, "batch": "accuracy",
                     "status": f"failed: {type(e).__name__}: {str(e)[:100]}"})
        print(rows[-1])

with open("lstm_gpu.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=sorted({k for r in rows for k in r}))
    w.writeheader(); w.writerows(rows)
print("Saved lstm_gpu.csv")

torch 2.14.1+cu130 | GPU: Tesla T4 | vocab: 66650
params: 16880350
FLOPs/sample: 25995000
{'device': 'cuda', 'precision': 'fp32', 'batch': 1, 'reps': 100, 'median_s': 0.000864227500017023, 'std_s': 2.3041312013623473e-05, 'p95_s': 0.000912594000055833, 'p99_s': 0.0009666739999829588, 'throughput_seq_s': 1157.1027304503764, 'gflops_per_s': 30.078885478057533, 'status': 'ok'}
{'device': 'cuda', 'precision': 'fp32', 'batch': 2, 'reps': 10, 'median_s': 0.0009518050000565381, 'std_s': 1.586893639659808e-05, 'p95_s': 0.0009638770000037766, 'p99_s': 0.0009638770000037766, 'throughput_seq_s': 2101.270743357303, 'gflops_per_s': 54.622532973573094, 'status': 'ok'}
{'device': 'cuda', 'precision': 'fp32', 'batch': 4, 'reps': 10, 'median_s': 0.0011450284999909854, 'std_s': 8.908326393450755e-06, 'p95_s': 0.001156465000008211, 'p99_s': 0.001156465000008211, 'throughput_seq_s': 3493.362828987655, 'gflops_per_s': 90.80996673953409, 'status': 'ok'}
{'device': 'cuda', 'precision': 'fp32', 'batch': 8, 'r

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…): reconstructing file:   0%|          |  0.00B /  733kB            

wikitext-2-raw-v1/test-00000-of-00001.pa(…): downloading bytes:           |  0.00B            

wikitext-2-raw-v1/train-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 6.36MB            

wikitext-2-raw-v1/train-00000-of-00001.p(…): downloading bytes:           |  0.00B            

... (2 outputs left out)

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

accuracy windows: 10000
fp32 accuracy: 0.22
fp16 accuracy: 0.2207
bf16 accuracy: 0.2204
Saved lstm_gpu.csv


# TPU Inference Testing

In [ ]:
!pip -q install datasets

import torch, torch.nn as nn, time, statistics, csv, torch_tpu
from datasets import load_dataset

torch.manual_seed(0)
dev = torch.device("tpu")
WIN = 20
ck = torch.load("lstm_wikitext2.pt", map_location="cpu")
vocab = ck["vocab"]; V = len(vocab)
print("torch", torch.__version__, "| device:", dev, "| vocab:", V)

class LSTM(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embedded = nn.Embedding(vocab_size, 100)
        self.lstm = nn.LSTM(100, 150, batch_first=True)
        self.linear = nn.Linear(150, vocab_size)
    def forward(self, x):
        emb = self.embedded(x)
        _, (final_hidden, _) = self.lstm(emb)
        return self.linear(final_hidden.squeeze(0))

def load_model(dt):
    m = LSTM(V); m.load_state_dict(ck["state_dict"]); m.eval()
    return m.to(dt).to(dev)

def sync(out):
    out.sum().item()          # forces the TPU to finish, brings a value back to host

print("params:", sum(p.numel() for p in LSTM(V).parameters()))
H, I = 150, 100
macs = 4 * H * (I + H) * WIN + H * V
flops_per_sample = 2 * macs
print("FLOPs/sample:", flops_per_sample)

def make_inputs(bs):
    return torch.randint(1, V, (bs, WIN), device=dev)   # Long ids

compile_times = {}
def bench(m, name, dt, bs, reps):
    x = make_inputs(bs)
    with torch.inference_mode():
        t0 = time.perf_counter(); sync(m(x))            # first call at this shape = compile
        compile_times[(name, bs)] = time.perf_counter() - t0
        for _ in range(2):
            sync(m(x))
        ts = []
        for _ in range(reps):
            t0 = time.perf_counter(); out = m(x); sync(out); ts.append(time.perf_counter() - t0)
    s = sorted(ts); med = statistics.median(ts)
    return {"device": "tpu", "precision": name, "batch": bs, "reps": reps,
            "median_s": med, "std_s": statistics.stdev(ts),
            "p95_s": s[int(0.95*(len(s)-1))], "p99_s": s[int(0.99*(len(s)-1))],
            "throughput_seq_s": bs / med,
            "gflops_per_s": flops_per_sample * bs / med / 1e9,
            "compile_s": compile_times[(name, bs)], "status": "ok"}

rows = []
for name, dt in [("fp32", torch.float32), ("bf16", torch.bfloat16)]:
    try:
        m = load_model(dt)
    except Exception as e:
        rows.append({"device": "tpu", "precision": name, "status": f"load failed: {type(e).__name__}: {str(e)[:100]}"})
        print(rows[-1]); continue
    for bs in [1, 2, 4, 8, 16, 32]:
        try:
            rows.append(bench(m, name, dt, bs, 100 if bs == 1 else 10))
        except Exception as e:
            rows.append({"device": "tpu", "precision": name, "batch": bs,
                         "status": f"failed: {type(e).__name__}: {str(e)[:100]}"})
        print(rows[-1])

# ---------- accuracy on the same 10,000 validation windows ----------
val = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="validation")
vt = []
for line in val["text"]:
    vt.extend(line.lower().split())
ids = [vocab.get(w, 0) for w in vt]
X = torch.tensor([ids[i:i+WIN] for i in range(len(ids)-WIN)])
y = torch.tensor([ids[i+WIN] for i in range(len(ids)-WIN)])
Xs, ys = X[:10000], y[:10000]
print("accuracy windows:", len(Xs))

for name, dt in [("fp32", torch.float32), ("bf16", torch.bfloat16)]:
    try:
        m = load_model(dt)
        correct = 0
        with torch.inference_mode():
            for i in range(0, len(Xs), 256):
                out = m(Xs[i:i+256].to(dev)).float().cpu()
                correct += (out.argmax(1) == ys[i:i+256]).sum().item()
        acc = correct / len(Xs)
        rows.append({"device": "tpu", "precision": name, "batch": "accuracy",
                     "reps": len(Xs), "accuracy": acc, "status": "ok"})
        print(name, "accuracy:", acc)
    except Exception as e:
        rows.append({"device": "tpu", "precision": name, "batch": "accuracy",
                     "status": f"failed: {type(e).__name__}: {str(e)[:100]}"})
        print(rows[-1])

with open("lstm_tpu.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=sorted({k for r in rows for k in r}))
    w.writeheader(); w.writerows(rows)
print("Saved lstm_tpu.csv")